# <center> <img src="../img/ITESOLogo.png" alt="ITESO" width="480" height="130"> </center>
# <center> **Departamento de Electrónica, Sistemas e Informática** </center>
---
## <center> **Big Data** </center>
---
### <center> **Spring 2026** </center>
---
### <center> **Examples on Data Joins and JSON columns** </center>
---
**Profesor**: Pablo Camarillo Ramirez

# Create SparkSession

In [10]:
from SparkUtils import SparkUtils

from pyspark.sql.functions import get_json_object, from_json

In [5]:
# create Spark session
MASTER_URL = "spark://spark-master:7077"
APP_NAME = "Example - Extract info from JSON"

spark = SparkUtils(MASTER_URL, APP_NAME)._spark

spark

# Manipulating JSON Columns

In [6]:
schema = SparkUtils\
    .generate_schema([("id", "int"), ("json_col", "string")])

data = [
    (1, '{"name": "Alice", "age": 25, "payments": [34, 433, 54], "address": {"city": "New York", "zip": "10001"}}'),
    (2, '{"name": "Bob", "age": 30, "address": {"city": "Los Angeles", "zip": "90001"}}'),
    (3, '{"name": "Charlie", "age": 35, "address": {"city": "Chicago", "zip": "60601"}}')
]

df = spark.createDataFrame(data, schema)

df.show(truncate=False)

+---+--------------------------------------------------------------------------------------------------------+
|id |json_col                                                                                                |
+---+--------------------------------------------------------------------------------------------------------+
|1  |{"name": "Alice", "age": 25, "payments": [34, 433, 54], "address": {"city": "New York", "zip": "10001"}}|
|2  |{"name": "Bob", "age": 30, "address": {"city": "Los Angeles", "zip": "90001"}}                          |
|3  |{"name": "Charlie", "age": 35, "address": {"city": "Chicago", "zip": "60601"}}                          |
+---+--------------------------------------------------------------------------------------------------------+



### Extract a JSON column with get_json_object function

In [8]:
df = df.withColumn("name", get_json_object(df.json_col, "$.name"))

df.show()

+---+--------------------+-------+
| id|            json_col|   name|
+---+--------------------+-------+
|  1|{"name": "Alice",...|  Alice|
|  2|{"name": "Bob", "...|    Bob|
|  3|{"name": "Charlie...|Charlie|
+---+--------------------+-------+



In [9]:
df = df.withColumn("age", get_json_object(df.json_col, "$.age"))

df.show()

+---+--------------------+-------+---+
| id|            json_col|   name|age|
+---+--------------------+-------+---+
|  1|{"name": "Alice",...|  Alice| 25|
|  2|{"name": "Bob", "...|    Bob| 30|
|  3|{"name": "Charlie...|Charlie| 35|
+---+--------------------+-------+---+



In [13]:
df = df\
    .withColumn("city", get_json_object(df.json_col, "$.address.city"))

df.show()

+---+--------------------+-------+---+-----------+
| id|            json_col|   name|age|       city|
+---+--------------------+-------+---+-----------+
|  1|{"name": "Alice",...|  Alice| 25|   New York|
|  2|{"name": "Bob", "...|    Bob| 30|Los Angeles|
|  3|{"name": "Charlie...|Charlie| 35|    Chicago|
+---+--------------------+-------+---+-----------+



### Extact a JSON column with from_json

In [ ]:
# define the schema of the JSON object
people_schema = SparkUtils\
    .generate_schema([
        ("name", "string"),
        ("age", "int"),
        ("payments", "array_int"),
        ("address", "struct")
    ])

df_parsed = df\
    .withColumn("parsed", from_json(df.json_col, people_schema))

df_parsed.printSchema()

df_parsed.show(truncate=False)

root
 |-- id: integer (nullable = true)
 |-- json_col: string (nullable = true)
 |-- name: string (nullable = true)
 |-- age: string (nullable = true)
 |-- city: string (nullable = true)
 |-- parsed: struct (nullable = true)
 |    |-- name: string (nullable = true)
 |    |-- age: integer (nullable = true)
 |    |-- payments: array (nullable = true)
 |    |    |-- element: integer (containsNull = true)
 |    |-- address: struct (nullable = true)



+---+--------------------------------------------------------------------------------------------------------+-------+---+-----------+------------------------------+
|id |json_col                                                                                                |name   |age|city       |parsed                        |
+---+--------------------------------------------------------------------------------------------------------+-------+---+-----------+------------------------------+
|1  |{"name": "Alice", "age": 25, "payments": [34, 433, 54], "address": {"city": "New York", "zip": "10001"}}|Alice  |25 |New York   |{Alice, 25, [34, 433, 54], {}}|
|2  |{"name": "Bob", "age": 30, "address": {"city": "Los Angeles", "zip": "90001"}}                          |Bob    |30 |Los Angeles|{Bob, 30, NULL, {}}           |
|3  |{"name": "Charlie", "age": 35, "address": {"city": "Chicago", "zip": "60601"}}                          |Charlie|35 |Chicago    |{Charlie, 35, NULL, {}}       |
+---

In [ ]:
from pyspark.sql.functions import col
df_parsed.select(col("parsed.name"), col("parsed.payments").getItem(1)).show()

## Data Join

In [ ]:
df_a = su._spark.createDataFrame([(1, "Alice"), (2, "Bob")],
                              ["id", "name"])
df_b = su._spark.createDataFrame([(3, "Charlie"), (4, "David")],
                              ["id", "name"])
result = df_a.union(df_b)
result.show()

### Union w/o duplicates

In [ ]:
df_a = su._spark.createDataFrame([(1, "Alice"), (2, "Bob")],
                              ["id", "name"])
df_b = su._spark.createDataFrame([(1, "Alice"), (4, "David")],
                              ["id", "name"])
df_a.union(df_b).distinct().show()

### Union with Mismatched Schemas

In [ ]:
df_a = su._spark.createDataFrame([(1, "Alice")], ["id", "name"])
df_b = su._spark.createDataFrame([("Bob", 2)], ["name", "id"])
result = df_a.unionByName(df_b)
result.show()

### Union by Name with missing columns

In [ ]:
df_a = su._spark.createDataFrame([(1, "Alice", "NY")], ["id", "name", "city"])
df_a.show()
df_b = su._spark.createDataFrame([(2, "Bob")], ["id", "name"])
df_b.show()
result = df_a.unionByName(df_b, allowMissingColumns=True)
result.show()

## Left Join

### Datasets

In [15]:
book_data = [
    ("Game of thrones", 400, 1),
    ("Spark", 500, 2),
    ("Kafka", 300, 3),
    ("Java", 350, 5)
]

writer_data = [
    ("George R.R. Martin", 1),
    ("Zaharia", 2),
    ("Neha", 3),
    ("James", 4)
]

books_df = spark\
    .createDataFrame(book_data, ["book_name", "cost", "writer_id"])

writers_df = spark\
    .createDataFrame(writer_data, ["writer_name", "writer_id"])

books_df.show()
writers_df.show()

+---------------+----+---------+
|      book_name|cost|writer_id|
+---------------+----+---------+
|Game of thrones| 400|        1|
|          Spark| 500|        2|
|          Kafka| 300|        3|
|           Java| 350|        5|
+---------------+----+---------+



+------------------+---------+
|       writer_name|writer_id|
+------------------+---------+
|George R.R. Martin|        1|
|           Zaharia|        2|
|              Neha|        3|
|             James|        4|
+------------------+---------+



In [16]:
result = books_df\
      .join(
          writers_df,
          books_df["writer_id"] == writers_df["writer_id"],
          "left"
      )

result.show()

+---------------+----+---------+------------------+---------+
|      book_name|cost|writer_id|       writer_name|writer_id|
+---------------+----+---------+------------------+---------+
|Game of thrones| 400|        1|George R.R. Martin|        1|
|          Spark| 500|        2|           Zaharia|        2|
|           Java| 350|        5|              NULL|     NULL|
|          Kafka| 300|        3|              Neha|        3|
+---------------+----+---------+------------------+---------+



In [17]:
result = books_df\
    .join(
        writers_df,
        on="writer_id",
        how="left"
    )

result.show()

+---------+---------------+----+------------------+
|writer_id|      book_name|cost|       writer_name|
+---------+---------------+----+------------------+
|        1|Game of thrones| 400|George R.R. Martin|
|        2|          Spark| 500|           Zaharia|
|        5|           Java| 350|              NULL|
|        3|          Kafka| 300|              Neha|
+---------+---------------+----+------------------+



## Right join

In [18]:
result = books_df\
    .join(
        writers_df,
        books_df["writer_id"] == writers_df["writer_id"],
        "right"
    )

result.show()

+---------------+----+---------+------------------+---------+
|      book_name|cost|writer_id|       writer_name|writer_id|
+---------------+----+---------+------------------+---------+
|Game of thrones| 400|        1|George R.R. Martin|        1|
|          Spark| 500|        2|           Zaharia|        2|
|          Kafka| 300|        3|              Neha|        3|
|           NULL|NULL|     NULL|             James|        4|
+---------------+----+---------+------------------+---------+



In [19]:
result = books_df\
    .join(
        writers_df,
        on="writer_id",
        how="right"
    )

result.show()

+---------+---------------+----+------------------+
|writer_id|      book_name|cost|       writer_name|
+---------+---------------+----+------------------+
|        1|Game of thrones| 400|George R.R. Martin|
|        2|          Spark| 500|           Zaharia|
|        3|          Kafka| 300|              Neha|
|        4|           NULL|NULL|             James|
+---------+---------------+----+------------------+



### Inner Join

In [20]:
books_df\
    .join(
        writers_df,
        on="writer_id"
    ).show()

+---------+---------------+----+------------------+
|writer_id|      book_name|cost|       writer_name|
+---------+---------------+----+------------------+
|        1|Game of thrones| 400|George R.R. Martin|
|        2|          Spark| 500|           Zaharia|
|        3|          Kafka| 300|              Neha|
+---------+---------------+----+------------------+



In [21]:
spark.stop()